In [0]:
-- Revenue Analysis

DROP VIEW analysis.monthly_revenue;

CREATE OR REPLACE VIEW monthly_revenue AS
SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
    SUM(p.payment_value) AS revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM gold.fact_orders o
LEFT JOIN gold.fact_payments p
  ON o.order_id = p.order_id
GROUP BY month
ORDER BY month;

--Customer Metrics
DROP VIEW analysis.customer_metrics;

CREATE OR REPLACE VIEW customer_metrics AS
SELECT
    o.customer_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(p.payment_value) AS total_spent,
    AVG(p.payment_value) AS avg_order_value,
    MAX(o.order_purchase_timestamp) AS last_order_date
FROM gold.fact_orders o
JOIN gold.fact_payments p
    ON o.order_id = p.order_id
GROUP BY o.customer_id;

-- Delivery Performance
DROP VIEW analysis.delivery_performance_by_state;

CREATE OR REPLACE VIEW analysis.delivery_performance_by_state AS
SELECT
    c.customer_state,
    AVG(o.actual_delivery_days) AS avg_delivery_days,
    AVG(o.is_late) AS delay_rate
FROM gold.fact_orders o
JOIN gold.dim_customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state;

-- ML Feature
DROP VIEW analysis.ml_features;

CREATE OR REPLACE VIEW analysis.ml_features AS
SELECT
    o.order_id,
    o.customer_id,
    p.payment_value,
    p.payment_installments,
    p.payment_type,
    o.actual_delivery_days,
    o.estimated_delivery_days,
    o.is_late
FROM gold.fact_orders o
JOIN gold.fact_payments p
    ON o.order_id = p.order_id;